# 📊 Consolidação e Exportação da ABT Final (`abt_features_modelagem`)
**Projeto:** Sanidade-Vegetal (SugarVision)  
**Sprint:** 2 — Framework SEMMA (Fase: Modify)  
**Responsável:** Elisa (`EA`) — Engenharia de Machine Learning & MLOps  
**Dataset Consolidado:** `data/processed/abt_features_modelagem.parquet` (6.571 instâncias x 40 colunas)  

---

## 📌 Objetivo da Entrega

Processar o conjunto completo de **6.571 instâncias** através dos transformadores atômicos desenvolvidos na Fase Modify, consolidar a **Analytical Base Table (ABT) final de modelagem**, validar formalmente a **ausência total de valores nulos (0 NaNs)** e persistir em formatos de alta performance (**Apache Parquet** e **CSV**), gerando manifesto de integridade criptográfica SHA-256 e garantindo prontidão contratual para o `GridSearchCV` na **Sprint 3 (Modelagem SVM)**.

### ✅ Cobertura do Checklist da Tarefa:
1. **Executar o pipeline de pré-processamento** sobre a base completa com divisão determinística.
2. **Validar ausência de valores null/NaN** e integridade das colunas preditivas e alvo.
3. **Salvar a base tratada em formato Apache Parquet** (`data/processed/abt_features_modelagem.parquet`) e CSV.
4. **Documentar o hash de integridade** e o volume final de linhas e colunas geradas.
5. **Garantir que o artefato esteja 100% pronto** para ser consumido pelo `GridSearchCV` na Sprint 3.

In [1]:
# 1. Configuração do ambiente e importação das dependências
import os
import sys
import json
import time
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import pyarrow

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"Pandas version   : {pd.__version__}")
print(f"PyArrow version  : {pyarrow.__version__}")
print(f"Sklearn version  : {sklearn.__version__}")

Pandas version   : 3.0.6
PyArrow version  : 25.0.1
Sklearn version  : 1.9.1


---
## 1. Carga e Inspeção Estrutural da Base Apache Parquet

Carregamos o arquivo colunar de alta performance `data/processed/abt_features_modelagem.parquet`, gerado pelo pipeline atômico com compressão Snappy.

In [2]:
base_dir = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
parquet_path = base_dir / "data" / "processed" / "abt_features_modelagem.parquet"
csv_path = base_dir / "data" / "processed" / "abt_features_modelagem.csv"

# Carregar Parquet
t0 = time.perf_counter()
df_abt = pd.read_parquet(parquet_path)
t_load = time.perf_counter() - t0

print(f"• Dataset carregado em {t_load*1000:.2f} ms")
print(f"• Dimensões: {df_abt.shape[0]} linhas x {df_abt.shape[1]} colunas.")

# Exibição das primeiras linhas com alvos e primeiras variáveis transformadas
display(df_abt.head(5))

• Dataset carregado em 40.83 ms
• Dimensões: 6571 linhas x 40 colunas.


,sample_id,split_partition,class_label,target_binary,target_multiclass,mean_hue,std_saturation,exg_index,exr_index,rg_ratio,indice_clorose_necrose,hue_dispersion,glcm_contrast,glcm_homogeneity,glcm_dissimilarity,glcm_energy,indice_rugosidade_pustula,laplacian_var,size_kb,total_pixels,resolution_mp,aspect_ratio,dataset_source_mendeley_data,dataset_source_roboflow_sugarcane,extension_.jpeg,extension_.jpg,size_kb_bin_0,size_kb_bin_1,size_kb_bin_2,size_kb_bin_3,laplacian_var_bin_0,laplacian_var_bin_1,laplacian_var_bin_2,laplacian_var_bin_3,resolution_bin_0,resolution_bin_1,resolution_bin_2,aspect_ratio_bin_0,aspect_ratio_bin_1,aspect_ratio_bin_2
0,7064f596-14b4-45fc-875c-f2b0fdeceb9d,test,HEALTHY,0,1,1.554876,-1.711918,2.163672,-1.135561,-1.781395,0.065539,-0.723465,-1.580613,1.433097,-1.515923,1.433097,-1.048776,-0.815430,-0.324743,-0.323932,-0.32393,-0.237265,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,4623a0fc-bec7-4ad8-a5c2-88eea86f8a0e,test,HEALTHY,0,1,1.381143,-0.927537,2.013610,-1.471668,-1.893143,-1.127516,0.397470,-1.663628,1.806913,-1.707639,1.806913,-1.083498,0.009917,-0.321361,-0.323932,-0.32393,-0.237265,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,34c5670d-c4fd-4996-bf78-83764aa6e4e3,test,HEALTHY,0,1,1.596528,-1.229818,1.793987,-1.635375,-1.870547,-0.816227,0.080223,-1.706199,1.421094,-1.562534,1.421094,-1.066483,-0.995601,-0.318895,-0.323932,-0.32393,-0.237265,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,25032583-78af-426d-a4a4-5a49294a848b,test,HEALTHY,0,1,1.836575,-1.288724,1.518091,-1.709985,-1.767837,-0.608667,0.145047,-1.512499,1.549700,-1.541877,1.549700,-1.047203,0.068429,-0.319211,-0.323932,-0.32393,-0.237265,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,a252dc67-f042-4296-9d1a-0ea5d7c25224,test,HEALTHY,0,1,1.354836,-1.363132,0.852970,-1.808499,-1.474088,-0.091576,-0.290894,-1.277291,1.355934,-1.363961,1.355934,-0.992138,0.042746,-0.323959,-0.323932,-0.32393,-0.237265,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


---
## 2. Auditoria Estrita de Qualidade e Ausência de Valores Nulos (Checklist 2)

Verificamos a ausência absoluta de valores nulos (`NaN` / `None`) em todas as 40 colunas, a unicidade do identificador `sample_id` e a conformidade das partições experimentais.

In [3]:
# 1. Total de valores nulos
null_counts = df_abt.isna().sum()
total_nulls = null_counts.sum()

print(f"• Total de valores nulos no dataset: {total_nulls}")
assert total_nulls == 0, f"Existem valores nulos: {total_nulls}"
print("✓ [VALIDADO] 0 NaNs em 100% das 40 colunas da ABT final.")

# 2. Unicidade de sample_id
n_unique = df_abt['sample_id'].nunique()
print(f"• Unicidade de sample_id: {n_unique} / {len(df_abt)}")
assert n_unique == len(df_abt), "Existem chaves primárias duplicadas!"
print("✓ [VALIDADO] Integridade de identificadores únicos confirmada.")

# 3. Distribuição das Partições Experimentais
split_counts = df_abt['split_partition'].value_counts()
print("\nDistribuição por split_partition:")
for split_name, count in split_counts.items():
    print(f"  - {split_name:6s}: {count:5d} instâncias ({count/len(df_abt)*100:5.2f}%)")

• Total de valores nulos no dataset: 0
✓ [VALIDADO] 0 NaNs em 100% das 40 colunas da ABT final.
• Unicidade de sample_id: 6571 / 6571
✓ [VALIDADO] Integridade de identificadores únicos confirmada.

Distribuição por split_partition:
  - train :  5574 instâncias (84.83%)
  - valid :   617 instâncias ( 9.39%)
  - test  :   380 instâncias ( 5.78%)


---
## 3. Integridade das Colunas de Alvo (Target Supervisionado)

Validamos o balanceamento do target binário (Sadia vs Doente) e das 7 classes fitopatológicas nominais em cada partição.

In [4]:
# Tabela cruzada de partição por classe fitopatológica
cross_classes = pd.crosstab(df_abt['class_label'], df_abt['split_partition'], margins=True)
display(cross_classes)

# Distribuição do target binário
binary_dist = df_abt.groupby(['split_partition', 'target_binary']).size().unstack(fill_value=0)
binary_dist.columns = ['Sadia (0)', 'Doente (1)']
display(binary_dist)

split_partition,test,train,valid,All
class_label,,,,
GRASSY SHOOT,36,150,20,206
HEALTHY,46,1005,95,1146
LEAF SCALD,63,313,63,439
MOSAIC,93,1010,154,1257
RED ROT,49,1125,94,1268
RUST,51,915,91,1057
YELLOW LEAF,42,1056,100,1198
All,380,5574,617,6571


,Sadia (0),Doente (1)
split_partition,,
test,46,334
train,1005,4569
valid,95,522


---
## 4. Benchmark Comparativo de Desempenho de I/O: Apache Parquet vs CSV (Checklist 3)

Comparamos a eficiência de armazenamento em disco e o tempo de leitura entre os formatos Apache Parquet (colunar com compressão Snappy) e CSV (texto delimitado plano).

In [5]:
# Benchmark Parquet
t0 = time.perf_counter()
_ = pd.read_parquet(parquet_path)
t_pq = (time.perf_counter() - t0) * 1000

# Benchmark CSV
t0 = time.perf_counter()
_ = pd.read_csv(csv_path)
t_csv = (time.perf_counter() - t0) * 1000

size_pq_kb = parquet_path.stat().st_size / 1024
size_csv_kb = csv_path.stat().st_size / 1024
ratio_size = size_csv_kb / size_pq_kb
speedup = t_csv / t_pq if t_pq > 0 else 1.0

benchmark_summary = pd.DataFrame([
    {"Formato": "Apache Parquet", "Tamanho (KB)": round(size_pq_kb, 2), "Tamanho (MB)": round(size_pq_kb/1024, 2), "Tempo I/O (ms)": round(t_pq, 2), "Compressão": "Snappy Colunar"},
    {"Formato": "CSV (Texto Plano)", "Tamanho (KB)": round(size_csv_kb, 2), "Tamanho (MB)": round(size_csv_kb/1024, 2), "Tempo I/O (ms)": round(t_csv, 2), "Compressão": "Nenhuma (Raw)"}
])
display(benchmark_summary)

print(f"• Ganho de Armazenamento : O arquivo Parquet é {ratio_size:.2f}x mais compacto que o CSV.")
print(f"• Ganho de Velocidade    : A carga via Parquet é {speedup:.2f}x mais rápida que o CSV.")

,Formato,Tamanho (KB),Tamanho (MB),Tempo I/O (ms),Compressão
0,Apache Parquet,916.92,0.90,5.8,Snappy Colunar
1,CSV (Texto Plano),2989.66,2.92,45.9,Nenhuma (Raw)


• Ganho de Armazenamento : O arquivo Parquet é 3.26x mais compacto que o CSV.
• Ganho de Velocidade    : A carga via Parquet é 7.91x mais rápida que o CSV.


---
## 5. Hashes Criptográficos e Manifesto de Integridade (Checklist 4)

Para auditoria e conformidade MLOps, inspecionamos o manifesto JSON `data/processed/abt_features_modelagem_manifest.json` que registra os hashes SHA-256 de integridade e metadados dos arquivos.

In [6]:
manifest_path = base_dir / "data" / "processed" / "abt_features_modelagem_manifest.json"

with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest_data = json.load(f)

print(f"• Artefato             : {manifest_data['artifact_name']}")
print(f"• Responsável          : {manifest_data['responsible']}")
print(f"• Gerado em            : {manifest_data['generated_at']}")
print(f"• Volumetria Final     : {manifest_data['shape']['rows']} linhas x {manifest_data['shape']['columns']} colunas.")
print(f"\nHashes SHA-256 Registrados:")
print(f"  - Parquet : {manifest_data['files']['parquet']['sha256']}")
print(f"  - CSV     : {manifest_data['files']['csv']['sha256']}")

• Artefato             : abt_features_modelagem
• Responsável          : Elisa
• Gerado em            : 2026-09-18 09:55:23
• Volumetria Final     : 6571 linhas x 40 colunas.

Hashes SHA-256 Registrados:
  - Parquet : e6bd41ebff356e2c2e6be8985f0026dfb0a11364c4c4f729657223cf103c4340
  - CSV     : b37d65c2bd7928eb2e76948b3540542a17ecdc28b000cffa2e990731e4128edc


---
## 6. Prontidão para Modelagem na Sprint 3 com GridSearchCV (Checklist 5)

Demonstramos o consumo direto da base tratada para a **Sprint 3 (Modelagem Supervisionada)**, isolando `X_train`, `y_train` e executando uma rodada de calibração de hiperparâmetros com `GridSearchCV` e `SVC(kernel='rbf')`.

In [7]:
# 1. Isolar features preditivas e target
exclude_cols = ['sample_id', 'split_partition', 'class_label', 'target_binary', 'target_multiclass']
feature_cols = [c for c in df_abt.columns if c not in exclude_cols]

print(f"• Quantidade de features preditivas: {len(feature_cols)}")

# 2. Filtrar conjunto de treino estrito
train_mask = df_abt['split_partition'] == 'train'
df_train = df_abt[train_mask]

# Subamostragem balanceada para teste rápido
idx_0 = df_train[df_train['target_binary'] == 0].sample(n=250, random_state=42).index
idx_1 = df_train[df_train['target_binary'] == 1].sample(n=250, random_state=42).index
sample_train = df_train.loc[idx_0.union(idx_1)]

X_train_sample = sample_train[feature_cols].values
y_train_sample = sample_train['target_binary'].values

# 3. Grade de Hiperparâmetros para SVM RBF
param_grid = {
    'C': [0.1, 1.0, 10.0],
    'gamma': ['scale', 'auto']
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_svm = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=cv, scoring='f1', n_jobs=1)

t0 = time.perf_counter()
grid_svm.fit(X_train_sample, y_train_sample)
t_grid = time.perf_counter() - t0

print(f"✓ [SUCESSO] GridSearchCV executado em {t_grid:.2f}s!")
print(f"• Melhor F1-Score obtido : {grid_svm.best_score_:.4f}")
print(f"• Melhores Hiperparâmetros: {grid_svm.best_params_}")
print("\n✓ CONTRATO VALIDADO: A base 'abt_features_modelagem.parquet' está 100% pronta para a Sprint 3!")

• Quantidade de features preditivas: 35


✓ [SUCESSO] GridSearchCV executado em 0.09s!
• Melhor F1-Score obtido : 1.0000
• Melhores Hiperparâmetros: {'C': 0.1, 'gamma': 'scale'}

✓ CONTRATO VALIDADO: A base 'abt_features_modelagem.parquet' está 100% pronta para a Sprint 3!
